# Part 2a - Baseline Price Predictor

> Case study part: Part 2 of 3 (first of two notebooks)
> Required for: All students (3-credit and 4-credit)
> Role: You are a Real Estate Analyst at Quad Capital Partners (QCP).

In Part 1 we described the campus-town markets. In Part 2 we want to
predict the list price of any single-family home in QCP's footprint from
its observable features - the kind of automated valuation model (AVM) that
underpins Zillow's *Zestimate* and Redfin's *Estimate*.

This notebook builds the baseline model. It uses only structured tabular
features (location, beds, baths, square footage, lot size, age, parking, and
the anchoring `campus`). Part 2b will add NLP-derived features from the
listing description and show how much that improves the model.

By the end of this notebook you should be able to answer:

1. What does a defensible baseline price-prediction pipeline look like?
2. How well does it perform on a held-out sample (MAE, RMSE, R², MAPE)?
3. Where does it work well and where does it fail?


## ⚙️ Setup


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

pd.set_option("display.max_columns", 60)


## 📥 Load the Cleaned Dataset from Part 1

If you have not run Part 1, do so first - it produces `properties_clean.parquet`.


In [ ]:
DATA_DIR = "data"

df = pd.read_parquet(f"{DATA_DIR}/properties_clean.parquet")
print(f"{len(df):,} cleaned listings loaded")
df.head(3)


## 🧱 Choose the Target and Features

Our target is the listing price. We model `log(price)` instead of raw price
because real-estate prices are right-skewed and a log target makes the model's
percentage error roughly constant across price bands - which matches how
investors actually think ("within 10%" is more useful than "within $50,000").


Define the target and the two feature lists (numeric and categorical).

In [ ]:
TARGET = "price"

NUMERIC_FEATURES = [
    "beds", "baths", "living_area", "lot_size",
    "year_built", "reso_parking_capacity",
    "latitude", "longitude",
]
CATEGORICAL_FEATURES = [
    "address_state", "address_city", "address_zipcode",
    "campus",
    "reso_architectural_style",
]


Build the modeling frame and the log-transformed target.

In [ ]:
model_df = df.dropna(subset=[TARGET]).copy()
model_df["log_price"] = np.log1p(model_df[TARGET])

X = model_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
# Zip code is numeric in the source but should be treated as a category.
X["address_zipcode"] = X["address_zipcode"].astype(str)
y_log = model_df["log_price"]
y_raw = model_df[TARGET]

X.head(3)


## ✂️ Train / Test Split

We hold out 20% of listings as a test set. We stratify by state so every state
appears in both splits in roughly the same proportion (otherwise a small
market like NC could end up entirely in one split and skew the metrics).


In [ ]:
X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X, y_log, y_raw,
    test_size=0.20, random_state=42,
    stratify=model_df["address_state"],
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")


## 🔧 Build a Reproducible Pipeline

We bundle preprocessing and the model into a single `Pipeline` so the full
training step is one `.fit()` call and the inference step is one `.predict()`
call. This is the same pattern used in the *Australian Rental Market* case.


Numeric and categorical preprocessors.

In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_FEATURES),
    ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
])


Wrap it together with an XGBoost regressor.

In [ ]:
baseline_model = Pipeline([
    ("preprocess", preprocessor),
    ("regressor", XGBRegressor(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
    )),
])


:::{note} Why XGBoost?

Gradient-boosted trees are the workhorse model for tabular real-estate data.
They handle non-linear relationships (price-per-sqft is not linear in size),
they cope gracefully with missing values after imputation, and they need
almost no scaling. We pick `XGBoost` because it is the de-facto industry
standard for this kind of tabular regression, trains quickly via the
`hist` tree method, and uses all available CPU cores out of the box.
:::


In [ ]:
baseline_model.fit(X_train, y_train_log)


## 📏 Evaluate on the Held-out Test Set

The model predicts `log(price)`, so we exponentiate before computing
business-friendly error metrics in dollars.


In [ ]:
def evaluate(model, X_eval, y_eval_raw, label=""):
    pred_log = model.predict(X_eval)
    pred = np.expm1(pred_log)
    mae = mean_absolute_error(y_eval_raw, pred)
    rmse = np.sqrt(mean_squared_error(y_eval_raw, pred))
    r2 = r2_score(y_eval_raw, pred)
    mape = np.mean(np.abs((y_eval_raw - pred) / y_eval_raw)) * 100
    print(f"=== {label} ===")
    print(f"  MAE   ${mae:>12,.0f}")
    print(f"  RMSE  ${rmse:>12,.0f}")
    print(f"  R²     {r2:>12.3f}")
    print(f"  MAPE   {mape:>11.2f}%")
    return pred

baseline_pred_test = evaluate(baseline_model, X_test, y_test_raw, "Baseline - Test")


:::{tip} Talking the metrics in business language

- MAE: "On average, the model is off by \$X."
- RMSE: same idea, but penalizes a single very wrong prediction more
  than MAE does. Always >= MAE.
- R²: "The model explains X% of the variation in price."
- MAPE: "On average, the model is off by X%." This is the metric the
  Investment Committee will actually quote.
:::


## 🔍 Residual Analysis - Where Does the Model Fail?

A 12% MAPE on average could still hide a market where the model is
systematically wrong. We break the error down by market and price band.


Build the diagnostic frame and aggregate errors by market.

In [ ]:
diag = pd.DataFrame({
    "actual": y_test_raw.values,
    "predicted": baseline_pred_test,
    "market": model_df.loc[X_test.index, "market"].values,
    "campus": model_df.loc[X_test.index, "campus"].values,
})
diag["abs_error"] = (diag["actual"] - diag["predicted"]).abs()
diag["pct_error"] = (diag["actual"] - diag["predicted"]) / diag["actual"] * 100

by_market = (
    diag.groupby("market", as_index=False)
        .agg(n=("actual", "count"),
             median_abs_error=("abs_error", "median"),
             median_pct_error=("pct_error", lambda s: s.abs().median()))
        .sort_values("median_pct_error")
)
by_market


In [ ]:
fig = px.bar(
    by_market.sort_values("median_pct_error"),
    x="median_pct_error", y="market", orientation="h",
    title="Baseline model - median |% error| by market",
    labels={"median_pct_error": "Median |% error|", "market": ""},
    color="median_pct_error", color_continuous_scale="Reds",
    height=600,
)
fig.show()


Bucket residuals by price band to see where the model is biased.

In [ ]:
diag["price_band"] = pd.cut(
    diag["actual"],
    bins=[0, 200_000, 350_000, 500_000, 750_000, 1_000_000, np.inf],
    labels=["<$200K", "$200-350K", "$350-500K", "$500-750K", "$750K-1M", ">$1M"],
)

fig = px.box(
    diag, x="price_band", y="pct_error",
    title="Baseline model - % error by price band",
    labels={"price_band": "Actual price band",
            "pct_error": "(actual - predicted) / actual × 100%"},
)
fig.update_yaxes(range=[-100, 100])
fig.add_hline(y=0, line_dash="dot")
fig.show()


:::{caution} A common failure mode

Tree-based models tend to under-predict the very top of the price
distribution (you'll see this clearly in the `>$1M` box). That is fine for
QCP - they don't buy trophy homes - but it would matter for a luxury-focused
fund. Always check the slice of the data you actually care about.
:::


## 📊 Predicted vs. Actual

The classic AVM diagnostic chart. Points should hug the diagonal.


In [ ]:
plot_sample = diag.sample(n=min(3000, len(diag)), random_state=0)

fig = px.scatter(
    plot_sample, x="actual", y="predicted",
    color="market", opacity=0.5,
    title="Baseline model - predicted vs. actual list price",
    labels={"actual": "Actual list price (USD)",
            "predicted": "Predicted list price (USD)"},
)
lim = max(plot_sample["actual"].max(), plot_sample["predicted"].max())
fig.add_shape(type="line", x0=0, y0=0, x1=lim, y1=lim, line=dict(dash="dash"))
fig.update_xaxes(range=[0, 1_200_000])
fig.update_yaxes(range=[0, 1_200_000])
fig.show()


## 🌟 Feature Importance - Which Inputs Matter?

XGBoost exposes a built-in `feature_importances_` attribute. Combined with
the one-hot-encoded names from the preprocessor, we can build a clean
ranking.


First, recover the post-encoding feature names from the pipeline.

In [ ]:
ohe = baseline_model.named_steps["preprocess"].named_transformers_["cat"].named_steps["ohe"]
cat_names = list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + cat_names


Then plot the top-20 most important features.

In [ ]:
importances = pd.Series(
    baseline_model.named_steps["regressor"].feature_importances_,
    index=all_names,
).sort_values(ascending=False).head(20)

imp_df = importances.sort_values().reset_index()
imp_df.columns = ["feature", "importance"]

fig = px.bar(
    imp_df, x="importance", y="feature", orientation="h",
    title="Top-20 features by XGBoost importance",
    labels={"importance": "Importance", "feature": ""},
    height=600,
)
fig.show()


## 💾 Save the Baseline Model & Predictions

Part 2b will need the same train/test split so the comparison is honest. We
save the row indices, not the model itself, because the model is small and
quick to retrain.


In [ ]:
splits = pd.DataFrame({
    "row_index": list(X_train.index) + list(X_test.index),
    "split": ["train"] * len(X_train) + ["test"] * len(X_test),
})
splits.to_parquet(f"{DATA_DIR}/baseline_split.parquet", index=False)

baseline_metrics = {
    "MAE": float(mean_absolute_error(y_test_raw, baseline_pred_test)),
    "RMSE": float(np.sqrt(mean_squared_error(y_test_raw, baseline_pred_test))),
    "R2": float(r2_score(y_test_raw, baseline_pred_test)),
    "MAPE": float(np.mean(np.abs((y_test_raw - baseline_pred_test) / y_test_raw)) * 100),
}
print("Baseline test metrics:", baseline_metrics)


## 🧠 Investment-Committee Takeaways (Part 2a)

Suggested bullets to include in your slide deck:

- The baseline AVM hits roughly X% MAPE across the QCP footprint - about
  what an early-stage Zestimate clone achieves.
- Errors are largest in [market] and on trophy homes (>\$1M).
- The most predictive features are size (`living_area`), location
  (`campus`, `address_zipcode`, `latitude`, `longitude`), and `baths` -
  same as in the industry-standard hedonic model.

In Part 2b we will see whether the listing description (free text) can
push that error meaningfully lower.
